# Colab：MSD 六任务分割 Finetune（各 150 epoch，均从 `0.pth` 冷启动）

**路径约定（Drive）**

| 用途 | 路径 |
|------|------|
| 预训练权重 | `MyDrive/finetune/rawCheckPoint/0.pth` |
| MSD 数据 | `MyDrive/dataset/unzip/TaskXX_*` |
| 输出 | `MyDrive/finetune/output/task03/` … `task10/` |

每个任务单独一个代码单元；**互不 resume**，每次都从 `0.pth` 加载 ViT 初值。

**各任务单元内可调参数（显式写在对应 cell 顶部）**

| 变量 | 默认 | 说明 |
|------|------|------|
| `*_EPOCHS` | 150 | 训练轮数 |
| `*_VAL_EVERY` | 5 | 每隔多少 epoch 验证 |
| `*_WORKERS` | 0 | DataLoader 进程数 |
| `*_BATCH_SIZE` | 1 | **训练显存**：可先试 2 |
| `*_NUM_SAMPLES` | 4 | 每 volume 随机 crop 数 |
| `*_CACHE_NUM` | 1 | MONAI 缓存卷数（占 RAM） |
| `*_SW_BATCH_SIZE` | 4 | 验证 sliding window batch |
| `*_INFER_OVERLAP` | 0.75 | 验证 overlap |
| `*_EXTRA_ARGS` | `[]` | 其它 `main.py` 参数，如 `["--noamp"]` |

> `LIDCIDRI` 为 DICOM，非 MSD 布局，本笔记本不包含；需先预处理。

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Drive mounted.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ── 全局默认（各任务 cell 可单独覆盖 TASKxx_*）────────────────────────
GITHUB_REPO = "https://github.com/tanglehunter00/AR-SSL4M-DEMO.git"
REPO_DIR = Path("/content/AR-SSL4M-DEMO")
FINETUNE_ROOT = Path("/content/drive/MyDrive/finetune")
DATASET_UNZIP = Path("/content/drive/MyDrive/dataset/unzip")
PRETRAIN_CKPT = FINETUNE_ROOT / "rawCheckPoint" / "0.pth"
OUTPUT_ROOT = FINETUNE_ROOT / "output"
EPOCHS = 150       # 各任务 TASKxx_EPOCHS 默认引用此值
VAL_EVERY = 5      # 各任务 TASKxx_VAL_EVERY 默认引用此值
WORKERS = 0        # 各任务 TASKxx_WORKERS 默认引用此值；Colab RAM 紧张保持 0
# ─────────────────────────────────────────────────────────────────────

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    print("git pull:", REPO_DIR)
else:
    subprocess.check_call(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
    print("git clone:", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "monai", "tensorboardX", "timm", "transformers", "scipy", "h5py",
    "batchgenerators", "nibabel", "SimpleITK"])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("需要 GPU Runtime：运行时 → 更改运行时类型 → T4/A100 等")

assert PRETRAIN_CKPT.is_file(), f"缺少预训练权重: {PRETRAIN_CKPT}"
assert DATASET_UNZIP.is_dir(), f"缺少数据目录: {DATASET_UNZIP}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from newFineTune.colab_msd_finetune import run_one_task, MSD_FINETUNE_TASKS

print("CUDA:", torch.cuda.get_device_name(0))
print("Repo:", REPO_DIR)
print("Pretrain:", PRETRAIN_CKPT)
print("Tasks:", MSD_FINETUNE_TASKS)
print("Output root:", OUTPUT_ROOT)

## Task03 — Liver
输出 → `finetune/output/task03/`

In [ ]:
# ── Task03 Liver：本 cell 内按需修改 ─────────────────────────────────
TASK03_EPOCHS = EPOCHS
TASK03_VAL_EVERY = VAL_EVERY
TASK03_WORKERS = WORKERS
TASK03_BATCH_SIZE = 1          # 显存有余量可试 2
TASK03_NUM_SAMPLES = 4
TASK03_CACHE_NUM = 1
TASK03_SW_BATCH_SIZE = 4
TASK03_INFER_OVERLAP = 0.75
TASK03_EXTRA_ARGS = []         # 例: ["--noamp"]

run_one_task(
    task_slug="task03",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK03_EPOCHS,
    val_every=TASK03_VAL_EVERY,
    workers=TASK03_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK03_BATCH_SIZE),
        "--num_samples", str(TASK03_NUM_SAMPLES),
        "--cache_num", str(TASK03_CACHE_NUM),
        "--sw_batch_size", str(TASK03_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK03_INFER_OVERLAP),
        *TASK03_EXTRA_ARGS,
    ],
)

## Task06 — Lung

In [ ]:
# ── Task06 Lung：本 cell 内按需修改 ──────────────────────────────────
TASK06_EPOCHS = EPOCHS
TASK06_VAL_EVERY = VAL_EVERY
TASK06_WORKERS = WORKERS
TASK06_BATCH_SIZE = 1
TASK06_NUM_SAMPLES = 4
TASK06_CACHE_NUM = 1
TASK06_SW_BATCH_SIZE = 4
TASK06_INFER_OVERLAP = 0.75
TASK06_EXTRA_ARGS = []

run_one_task(
    task_slug="task06",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK06_EPOCHS,
    val_every=TASK06_VAL_EVERY,
    workers=TASK06_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK06_BATCH_SIZE),
        "--num_samples", str(TASK06_NUM_SAMPLES),
        "--cache_num", str(TASK06_CACHE_NUM),
        "--sw_batch_size", str(TASK06_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK06_INFER_OVERLAP),
        *TASK06_EXTRA_ARGS,
    ],
)

## Task07 — Pancreas

In [ ]:
# ── Task07 Pancreas：本 cell 内按需修改 ──────────────────────────────
TASK07_EPOCHS = EPOCHS
TASK07_VAL_EVERY = VAL_EVERY
TASK07_WORKERS = WORKERS
TASK07_BATCH_SIZE = 1
TASK07_NUM_SAMPLES = 4
TASK07_CACHE_NUM = 1
TASK07_SW_BATCH_SIZE = 4
TASK07_INFER_OVERLAP = 0.75
TASK07_EXTRA_ARGS = []

run_one_task(
    task_slug="task07",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK07_EPOCHS,
    val_every=TASK07_VAL_EVERY,
    workers=TASK07_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK07_BATCH_SIZE),
        "--num_samples", str(TASK07_NUM_SAMPLES),
        "--cache_num", str(TASK07_CACHE_NUM),
        "--sw_batch_size", str(TASK07_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK07_INFER_OVERLAP),
        *TASK07_EXTRA_ARGS,
    ],
)

## Task08 — Hepatic Vessel

In [ ]:
# ── Task08 HepaticVessel：本 cell 内按需修改 ─────────────────────────
TASK08_EPOCHS = EPOCHS
TASK08_VAL_EVERY = VAL_EVERY
TASK08_WORKERS = WORKERS
TASK08_BATCH_SIZE = 1
TASK08_NUM_SAMPLES = 4
TASK08_CACHE_NUM = 1
TASK08_SW_BATCH_SIZE = 4
TASK08_INFER_OVERLAP = 0.75
TASK08_EXTRA_ARGS = []

run_one_task(
    task_slug="task08",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK08_EPOCHS,
    val_every=TASK08_VAL_EVERY,
    workers=TASK08_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK08_BATCH_SIZE),
        "--num_samples", str(TASK08_NUM_SAMPLES),
        "--cache_num", str(TASK08_CACHE_NUM),
        "--sw_batch_size", str(TASK08_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK08_INFER_OVERLAP),
        *TASK08_EXTRA_ARGS,
    ],
)

## Task09 — Spleen

In [ ]:
# ── Task09 Spleen：本 cell 内按需修改 ───────────────────────────────
TASK09_EPOCHS = EPOCHS
TASK09_VAL_EVERY = VAL_EVERY
TASK09_WORKERS = WORKERS
TASK09_BATCH_SIZE = 1
TASK09_NUM_SAMPLES = 4
TASK09_CACHE_NUM = 1
TASK09_SW_BATCH_SIZE = 4
TASK09_INFER_OVERLAP = 0.75
TASK09_EXTRA_ARGS = []

run_one_task(
    task_slug="task09",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK09_EPOCHS,
    val_every=TASK09_VAL_EVERY,
    workers=TASK09_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK09_BATCH_SIZE),
        "--num_samples", str(TASK09_NUM_SAMPLES),
        "--cache_num", str(TASK09_CACHE_NUM),
        "--sw_batch_size", str(TASK09_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK09_INFER_OVERLAP),
        *TASK09_EXTRA_ARGS,
    ],
)

## Task10 — Colon

In [ ]:
# ── Task10 Colon：本 cell 内按需修改 ────────────────────────────────
TASK10_EPOCHS = EPOCHS
TASK10_VAL_EVERY = VAL_EVERY
TASK10_WORKERS = WORKERS
TASK10_BATCH_SIZE = 1
TASK10_NUM_SAMPLES = 4
TASK10_CACHE_NUM = 1
TASK10_SW_BATCH_SIZE = 4
TASK10_INFER_OVERLAP = 0.75
TASK10_EXTRA_ARGS = []

run_one_task(
    task_slug="task10",
    repo_root=REPO_DIR,
    dataset_unzip_base=DATASET_UNZIP,
    pretrain_ckpt=PRETRAIN_CKPT,
    output_root=OUTPUT_ROOT,
    epochs=TASK10_EPOCHS,
    val_every=TASK10_VAL_EVERY,
    workers=TASK10_WORKERS,
    extra_main_args=[
        "--batch_size", str(TASK10_BATCH_SIZE),
        "--num_samples", str(TASK10_NUM_SAMPLES),
        "--cache_num", str(TASK10_CACHE_NUM),
        "--sw_batch_size", str(TASK10_SW_BATCH_SIZE),
        "--infer_overlap", str(TASK10_INFER_OVERLAP),
        *TASK10_EXTRA_ARGS,
    ],
)